# Stage 3: Data Profiling and Cleaning Methodology

## Objective

The goal of this stage is to inspect the FSA sample data, identify data quality problems, and design cleaning rules that can later be automated in the full data pipeline.

We will use the 100 Westminster establishments collected during Stage 2 as a sample.

## Stage 3 Workflow

1. Load the raw FSA JSON data
2. Extract the establishment records
3. Convert the data into a Pandas DataFrame
4. Inspect the dataset structure
5. Analyse missing values
6. Check duplicate records
7. Inspect data types
8. Inspect categorical values
9. Check unusual or invalid values
10. Define cleaning rules
11. Apply the cleaning rules to the sample
12. Validate the cleaned dataset

The original raw JSON file will remain unchanged. Cleaning and transformations will be performed separately so that the raw source data is always preserved.

## Step 1: Load the Raw FSA Data

The raw FSA API response saved during Stage 2 is loaded from:

`data/raw/fsa_westminster_sample.json`

The JSON contains three main sections:

* `establishments`, the food businesses and inspection information
* `meta`, metadata about the API response
* `links`, API navigation information

For the data profiling process, we will primarily work with the `establishments` section.


In [2]:
import json
import pandas as pd

# Open the raw FSA sample saved from Stage 2
with open("data/raw/fsa_westminster_sample.json", "r") as file:
    raw_data = json.load(file)

# Check the top-level keys in the JSON
print(raw_data.keys())

dict_keys(['establishments', 'meta', 'links'])


## Step 2: Extract Establishments and Create a DataFrame

The FSA JSON contains metadata and API links as well as the establishment records.

For profiling and cleaning, we only need the records stored inside the `establishments` section.

These records will be converted into a Pandas DataFrame so we can inspect rows, columns, missing values, data types, duplicates, and other data quality issues.


In [4]:
# Extract the establishment records
establishments = raw_data["establishments"]

# Convert the records into a Pandas DataFrame
df = pd.DataFrame(establishments)

# Check the number of rows and columns
print(df.shape)

# Preview the first 5 rows
df.head()

(100, 25)


,AddressLine1,AddressLine2,AddressLine3,AddressLine4,BusinessName,BusinessType,BusinessTypeID,ChangesByServerID,Distance,FHRSID,...,NewRatingPending,Phone,PostCode,RatingDate,RatingKey,RatingValue,RightToReply,SchemeType,geocode,scores
0,,,,,(ki:ts),Retailers - other,4613,0,None,1916465,...,False,,W1U,2026-03-04T00:00:00,fhrs_5_en-gb,5,,FHRS,"{'longitude': None, 'latitude': None}","{'Hygiene': 0, 'Structural': 0, 'ConfidenceInM..."
1,,,,,*Eatlife,Other catering premises,7841,0,None,1301468,...,False,,W1H,2025-04-17T00:00:00,fhrs_5_en-gb,5,,FHRS,"{'longitude': None, 'latitude': None}","{'Hygiene': 0, 'Structural': 0, 'ConfidenceInM..."
2,3 BERKELEY STREET,,LONDON,,1 Hotel Mayfair,Restaurant/Cafe/Canteen,1,0,None,1688813,...,False,,W1J 8DJ,2026-01-21T00:00:00,fhrs_5_en-gb,5,,FHRS,"{'longitude': '-0.1422003', 'latitude': '51.50...","{'Hygiene': 5, 'Structural': 0, 'ConfidenceInM..."
3,1-4 LEICESTER SQUARE,,LONDON,,1 Leicester Square Rooftop Bar and Restaurant,Restaurant/Cafe/Canteen,1,0,None,1898862,...,False,,WC2H 7NA,2026-01-09T00:00:00,fhrs_5_en-gb,5,,FHRS,"{'longitude': '-0.1306709', 'latitude': '51.51...","{'Hygiene': 5, 'Structural': 5, 'ConfidenceInM..."
4,1 WARWICK STREET,,LONDON,,1 Warwick,Restaurant/Cafe/Canteen,1,0,None,1559410,...,False,,W1B 5LR,2025-04-29T00:00:00,fhrs_5_en-gb,5,,FHRS,"{'longitude': '-0.1372907', 'latitude': '51.51...","{'Hygiene': 0, 'Structural': 0, 'ConfidenceInM..."


## Step 3: Inspect the Dataset Structure

Before cleaning the data, we need to understand what information is available.

We will inspect the column names to identify the variables provided by the FSA API and decide which fields may be useful later.

In [5]:
# Display all column names
print(df.columns.tolist())

['AddressLine1', 'AddressLine2', 'AddressLine3', 'AddressLine4', 'BusinessName', 'BusinessType', 'BusinessTypeID', 'ChangesByServerID', 'Distance', 'FHRSID', 'LocalAuthorityBusinessID', 'LocalAuthorityCode', 'LocalAuthorityEmailAddress', 'LocalAuthorityName', 'LocalAuthorityWebSite', 'NewRatingPending', 'Phone', 'PostCode', 'RatingDate', 'RatingKey', 'RatingValue', 'RightToReply', 'SchemeType', 'geocode', 'scores']


## Step 4: Inspect Data Types and Non-Null Values

We need to understand how Pandas has interpreted each column.

This will help identify:

- columns with missing values
- incorrect data types
- numeric columns stored as text
- date columns that may need conversion
- nested JSON columns that may need flattening

In [6]:
# Show each column, its data type, and number of non-null values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   AddressLine1                100 non-null    str   
 1   AddressLine2                100 non-null    str   
 2   AddressLine3                100 non-null    str   
 3   AddressLine4                100 non-null    str   
 4   BusinessName                100 non-null    str   
 5   BusinessType                100 non-null    str   
 6   BusinessTypeID              100 non-null    int64 
 7   ChangesByServerID           100 non-null    int64 
 8   Distance                    0 non-null      object
 9   FHRSID                      100 non-null    int64 
 10  LocalAuthorityBusinessID    100 non-null    str   
 11  LocalAuthorityCode          100 non-null    str   
 12  LocalAuthorityEmailAddress  100 non-null    str   
 13  LocalAuthorityName          100 non-null    str   
 14  LocalA

## Step 5: Analyse Missing Values

The dataset may contain both standard null values and empty strings.

Pandas `df.info()` detects values such as `None` and `NaN`, but an empty string (`""`) is still treated as a value.

We therefore need to check both types of missing data before deciding how each column should be cleaned.

In [7]:
# Count standard missing values such as None and NaN
null_counts = df.isna().sum()

# Count empty strings in text columns
empty_string_counts = (df == "").sum()

# Combine both checks into one table
missing_summary = pd.DataFrame({
    "null_values": null_counts,
    "empty_strings": empty_string_counts
})

# Show only columns that contain missing data
missing_summary[
    (missing_summary["null_values"] > 0) |
    (missing_summary["empty_strings"] > 0)
]

,null_values,empty_strings
AddressLine1,0,3
AddressLine2,0,97
AddressLine3,0,3
AddressLine4,0,100
Distance,100,0
Phone,0,100
PostCode,0,1
RightToReply,0,100


### Missing Value Findings

The initial profiling identified several columns containing missing information.

- `Distance` is completely null in the sample.
- `AddressLine4` is empty for all 100 establishments.
- `Phone` is empty for all 100 establishments.
- `RightToReply` is empty for all 100 establishments.
- `AddressLine2` is empty for 97 of the 100 establishments.
- `AddressLine1` and `AddressLine3` each contain 3 empty values.
- `PostCode` contains 1 empty value.

These findings will be used later to decide whether columns should be removed, retained, combined, or handled using specific missing-value rules.

No columns will be removed at this stage because the purpose of profiling is first to understand the data before defining the cleaning methodology.

## Step 6: Check for Duplicate Records

Duplicate records can distort statistical analysis and machine learning models.

We will first check whether any complete rows are duplicated in the sample.

In [9]:
# Check whether the same FSA establishment ID appears more than once
duplicate_fhrsid = df.duplicated(subset=["FHRSID"]).sum()

print("Duplicate FHRSID records:", duplicate_fhrsid)

Duplicate FHRSID records: 0


In [10]:
# Count each unique food hygiene rating
df["RatingValue"].value_counts(dropna=False)

RatingValue
5    100
Name: count, dtype: int64

## Step 8: Inspect Rating Dates

`RatingDate` records when each food hygiene rating was issued.

The column is currently stored as text, so we need to inspect its values before converting it into a proper datetime format.

We will check the range of dates and look for unusual or missing values.

In [11]:
# Preview the rating dates
print(df["RatingDate"].head(10))

# Check the earliest and latest values as currently stored
print("Minimum RatingDate:", df["RatingDate"].min())
print("Maximum RatingDate:", df["RatingDate"].max())

0    2026-03-04T00:00:00
1    2025-04-17T00:00:00
2    2026-01-21T00:00:00
3    2026-01-09T00:00:00
4    2025-04-29T00:00:00
5    2025-09-13T00:00:00
6    2024-05-07T00:00:00
7    2025-09-26T00:00:00
8    2023-12-01T00:00:00
9    2025-05-14T00:00:00
Name: RatingDate, dtype: str
Minimum RatingDate: 2018-03-20T00:00:00
Maximum RatingDate: 2026-07-25T00:00:00


### Rating Date Findings

`RatingDate` contains inspection dates ranging from March 2018 to July 2026.

The values appear to use a consistent ISO datetime format, but Pandas currently stores the column as text.

During the cleaning stage, this column should be converted to a proper datetime data type.

## Step 9: Inspect Categorical Values

Categorical columns should be inspected for inconsistent labels, unexpected categories, spelling differences, and rare values.

We will begin by examining `BusinessType`, `SchemeType`, and `RatingValue`.

In [14]:
# Check the different business types
print("Business Types:")
print(df["BusinessType"].value_counts())

print("\nScheme Types:")
print(df["SchemeType"].value_counts())

print("\nRating Values:")
print(df["RatingValue"].value_counts(dropna=False))

Business Types:
BusinessType
Restaurant/Cafe/Canteen                  65
Retailers - other                        12
Other catering premises                   6
Hotel/bed & breakfast/guest house         6
Pub/bar/nightclub                         4
School/college/university                 3
Takeaway/sandwich shop                    3
Retailers - supermarkets/hypermarkets     1
Name: count, dtype: int64

Scheme Types:
SchemeType
FHRS    100
Name: count, dtype: int64

Rating Values:
RatingValue
5    100
Name: count, dtype: int64


### Categorical Data Findings

The sample contains eight different business types, with `Restaurant/Cafe/Canteen` being the most common category.

All 100 establishments use the `FHRS` scheme, meaning `SchemeType` has no variation within this sample.

A significant limitation was identified in `RatingValue`. All 100 establishments have a hygiene rating of `5`.

This means the current 100-row sample is suitable for developing and testing the data cleaning methodology, but it is not suitable for training a classification model because the target variable contains only one class.

When the dataset is expanded later, the extraction process should include establishments with a wider range of hygiene ratings.

## Step 10: Inspect Nested JSON Fields

The FSA API contains nested data inside the `geocode` and `scores` columns.

These columns currently contain Python dictionaries rather than simple values.

Before flattening them into separate columns, we will inspect their structure and identify which fields may be useful for cleaning, analysis, and machine learning.

In [15]:
# Inspect the first geocode dictionary
print("Geocode example:")
print(df["geocode"].iloc[0])

# Inspect the first scores dictionary
print("\nScores example:")
print(df["scores"].iloc[0])

Geocode example:
{'longitude': None, 'latitude': None}

Scores example:
{'Hygiene': 0, 'Structural': 0, 'ConfidenceInManagement': 0}


## Step 11: Check Nested Field Consistency

Before converting the nested `geocode` and `scores` dictionaries into separate columns, we need to check whether their internal structure is consistent across the dataset.

This helps identify missing coordinates, missing inspection scores, or unexpected JSON structures before transformations are applied.

In [16]:
# Check which keys appear inside the geocode dictionaries
print("Geocode structures:")
print(df["geocode"].apply(lambda x: tuple(x.keys()) if isinstance(x, dict) else None).value_counts())

# Check which keys appear inside the scores dictionaries
print("\nScore structures:")
print(df["scores"].apply(lambda x: tuple(x.keys()) if isinstance(x, dict) else None).value_counts())

Geocode structures:
geocode
(longitude, latitude)    100
Name: count, dtype: int64

Score structures:
scores
(Hygiene, Structural, ConfidenceInManagement)    100
Name: count, dtype: int64


### Nested Field Findings

The nested JSON structure is consistent across all 100 establishments.

Every `geocode` record contains:

- `longitude`
- `latitude`

Every `scores` record contains:

- `Hygiene`
- `Structural`
- `ConfidenceInManagement`

This means these fields can be safely flattened into individual columns for the sample.

The inspection scores will be retained for analysis, but their use as machine learning features must be reviewed carefully because they may directly contribute to the final hygiene rating and therefore create target leakage.

## Step 12: Flatten Nested Fields

The `geocode` and `scores` columns contain useful information stored inside dictionaries.

We will extract these values into individual columns so they can be inspected and processed like normal tabular variables.

In [17]:
# Create a working copy so the original DataFrame remains unchanged
df_working = df.copy()

# Extract latitude and longitude
df_working["longitude"] = df_working["geocode"].apply(
    lambda x: x.get("longitude") if isinstance(x, dict) else None
)

df_working["latitude"] = df_working["geocode"].apply(
    lambda x: x.get("latitude") if isinstance(x, dict) else None
)

# Extract the three inspection scores
df_working["Hygiene"] = df_working["scores"].apply(
    lambda x: x.get("Hygiene") if isinstance(x, dict) else None
)

df_working["Structural"] = df_working["scores"].apply(
    lambda x: x.get("Structural") if isinstance(x, dict) else None
)

df_working["ConfidenceInManagement"] = df_working["scores"].apply(
    lambda x: x.get("ConfidenceInManagement") if isinstance(x, dict) else None
)

# Check the extracted columns
df_working[
    [
        "latitude",
        "longitude",
        "Hygiene",
        "Structural",
        "ConfidenceInManagement"
    ]
].head()

,latitude,longitude,Hygiene,Structural,ConfidenceInManagement
0,NaN,NaN,0.0,0.0,0.0
1,NaN,NaN,0.0,0.0,0.0
2,51.5076479,-0.1422003,5.0,0.0,5.0
3,51.510837,-0.1306709,5.0,5.0,0.0
4,51.5107175,-0.1372907,0.0,0.0,0.0


## Step 13: Profile Flattened Geographic and Inspection Fields

The nested fields have now been converted into individual columns.

Although the JSON structure was consistent across all establishments, the extracted data shows that some coordinate values may still be missing.

We will therefore inspect missing values and the distributions of the extracted geographic and inspection score fields before defining cleaning rules.


In [18]:
# Check missing values in the newly extracted columns
nested_missing = df_working[
    [
        "latitude",
        "longitude",
        "Hygiene",
        "Structural",
        "ConfidenceInManagement"
    ]
].isna().sum()

print("Missing values:")
print(nested_missing)

# Check the values appearing in each inspection score
print("\nHygiene scores:")
print(df_working["Hygiene"].value_counts(dropna=False))

print("\nStructural scores:")
print(df_working["Structural"].value_counts(dropna=False))

print("\nConfidence in Management scores:")
print(df_working["ConfidenceInManagement"].value_counts(dropna=False))

Missing values:
latitude                  6
longitude                 6
Hygiene                   3
Structural                3
ConfidenceInManagement    3
dtype: int64

Hygiene scores:
Hygiene
0.0    55
5.0    42
NaN     3
Name: count, dtype: int64

Structural scores:
Structural
5.0    63
0.0    34
NaN     3
Name: count, dtype: int64

Confidence in Management scores:
ConfidenceInManagement
0.0    51
5.0    46
NaN     3
Name: count, dtype: int64


### Flattened Field Findings

After flattening the nested fields:

- 6 establishments are missing latitude and longitude values.
- 3 establishments are missing `Hygiene`, `Structural`, and `ConfidenceInManagement` scores.
- The observed inspection score values in this sample are `0` and `5`.
- Missing inspection scores will not be imputed until their meaning and appropriate treatment are confirmed.

The inspection score fields will also require careful consideration before machine learning because they may create target leakage.

## Step 14: Inspect Postcodes

Postcodes will be important for geographic analysis and for joining the FSA data with external datasets such as ONS data.

We therefore need to inspect missing values, formatting, and unusual postcode values before defining the postcode cleaning rules.

In [19]:
# Show all postcode values
print(df_working["PostCode"].value_counts(dropna=False))

# Show records where the postcode is empty
df_working[df_working["PostCode"] == ""][
    ["BusinessName", "AddressLine1", "AddressLine2", "AddressLine3", "PostCode"]
]

PostCode
W1U         2
SW1H 0AZ    2
W2 2RY      2
W1H         1
W1J 8DJ     1
           ..
W1B 5PW     1
W1A 1AB     1
W2 3BP      1
W2 4UL      1
W1F 7TU     1
Name: count, Length: 97, dtype: int64


,BusinessName,AddressLine1,AddressLine2,AddressLine3,PostCode
39,A2Dominion Group,BEETHOVEN CENTRE THIRD AVENUE,,LONDON,


### Postcode Findings

The sample contains one establishment with a missing postcode.

The missing postcode belongs to `A2Dominion Group`.

Most postcode values appear to follow standard UK postcode formatting, although postcode formatting and validity should be standardised and validated during cleaning.

Postcodes will be retained because they may later be used for geographic analysis and joining FSA establishments with external datasets such as ONS data.

## Step 15: Inspect Potentially Redundant Columns

Some API fields may contain no useful information for analysis or machine learning.

We will inspect columns that appeared completely empty, constant, or primarily related to API administration before deciding whether they should be retained or removed.

In [20]:
# Inspect columns that may contain little or no useful information
columns_to_inspect = [
    "Distance",
    "Phone",
    "RightToReply",
    "ChangesByServerID",
    "LocalAuthorityCode",
    "LocalAuthorityName",
    "SchemeType"
]

for column in columns_to_inspect:
    print(f"\n{column}:")
    print(df_working[column].value_counts(dropna=False))


Distance:
Distance
None    100
Name: count, dtype: int64

Phone:
Phone
    100
Name: count, dtype: int64

RightToReply:
RightToReply
    100
Name: count, dtype: int64

ChangesByServerID:
ChangesByServerID
0    100
Name: count, dtype: int64

LocalAuthorityCode:
LocalAuthorityCode
533    100
Name: count, dtype: int64

LocalAuthorityName:
LocalAuthorityName
Westminster    100
Name: count, dtype: int64

SchemeType:
SchemeType
FHRS    100
Name: count, dtype: int64


### Redundant Column Findings

Several fields contain no variation or no usable information within the Westminster sample.

- `Distance` is missing for all establishments.
- `Phone` is empty for all establishments.
- `RightToReply` is empty for all establishments.
- `ChangesByServerID` contains the same value for every establishment.
- `LocalAuthorityCode` is constant because all records were extracted from Westminster.
- `SchemeType` is constant because all records use the FHRS scheme.

Completely empty fields are candidates for removal from the cleaned analytical dataset.

Constant administrative fields may still be useful for traceability or when the dataset is expanded nationally, so they should not automatically be removed from the raw data.

In [21]:
# Create an initial cleaning decision table
cleaning_decisions = pd.DataFrame({
    "column": [
        "Distance",
        "Phone",
        "RightToReply",
        "AddressLine4",
        "RatingDate",
        "geocode",
        "scores",
        "FHRSID",
        "BusinessType",
        "PostCode",
        "RatingValue",
        "Hygiene",
        "Structural",
        "ConfidenceInManagement"
    ],

    "treatment": [
        "Drop",
        "Drop",
        "Drop",
        "Drop",
        "Convert to datetime",
        "Flatten",
        "Flatten",
        "Keep as identifier",
        "Keep",
        "Standardise and keep",
        "Keep as target candidate",
        "Keep but exclude from initial ML",
        "Keep but exclude from initial ML",
        "Keep but exclude from initial ML"
    ]
})

cleaning_decisions

,column,treatment
0,Distance,Drop
1,Phone,Drop
2,RightToReply,Drop
3,AddressLine4,Drop
4,RatingDate,Convert to datetime
5,geocode,Flatten
6,scores,Flatten
7,FHRSID,Keep as identifier
8,BusinessType,Keep
9,PostCode,Standardise and keep


## Step 17: Define Cleaning Rules

Based on the profiling results, the following cleaning rules will be applied to the working dataset.

### Cleaning Rules

1. Drop columns that are completely empty and provide no analytical value:
   - `Distance`
   - `Phone`
   - `RightToReply`
   - `AddressLine4`

2. Convert empty strings in relevant text columns into proper missing values.

3. Convert `RatingDate` from text into a datetime data type.

4. Flatten `geocode` into:
   - `latitude`
   - `longitude`

5. Flatten `scores` into:
   - `Hygiene`
   - `Structural`
   - `ConfidenceInManagement`

6. Convert latitude and longitude into numeric data types.

7. Standardise postcode formatting by:
   - removing unnecessary spaces at the beginning or end
   - converting values to uppercase

8. Retain `FHRSID` as the unique establishment identifier.

9. Check for duplicate `FHRSID` records.

10. Retain missing coordinates rather than inventing geographic values.

11. Retain missing inspection scores rather than imputing values without justification.

12. Retain `RatingValue` as a possible machine learning target.

13. Exclude `Hygiene`, `Structural`, and `ConfidenceInManagement` from the initial machine learning feature set because they may create target leakage.

14. Preserve the original raw JSON file unchanged.

## Step 18: Apply the Cleaning Methodology

The cleaning rules defined during profiling will now be applied to the working DataFrame.

The original raw data and original DataFrame will remain unchanged.

In [22]:
import numpy as np

# Create a fresh copy of the original DataFrame
df_clean = df.copy()


# -----------------------------------
# 1. Replace empty strings with NaN
# -----------------------------------

df_clean = df_clean.replace("", np.nan)


# -----------------------------------
# 2. Flatten geocode
# -----------------------------------

df_clean["longitude"] = df_clean["geocode"].apply(
    lambda x: x.get("longitude") if isinstance(x, dict) else None
)

df_clean["latitude"] = df_clean["geocode"].apply(
    lambda x: x.get("latitude") if isinstance(x, dict) else None
)


# -----------------------------------
# 3. Flatten inspection scores
# -----------------------------------

df_clean["Hygiene"] = df_clean["scores"].apply(
    lambda x: x.get("Hygiene") if isinstance(x, dict) else None
)

df_clean["Structural"] = df_clean["scores"].apply(
    lambda x: x.get("Structural") if isinstance(x, dict) else None
)

df_clean["ConfidenceInManagement"] = df_clean["scores"].apply(
    lambda x: x.get("ConfidenceInManagement")
    if isinstance(x, dict)
    else None
)


# -----------------------------------
# 4. Convert RatingDate to datetime
# -----------------------------------

df_clean["RatingDate"] = pd.to_datetime(
    df_clean["RatingDate"],
    errors="coerce"
)


# -----------------------------------
# 5. Convert coordinates to numeric
# -----------------------------------

df_clean["latitude"] = pd.to_numeric(
    df_clean["latitude"],
    errors="coerce"
)

df_clean["longitude"] = pd.to_numeric(
    df_clean["longitude"],
    errors="coerce"
)


# -----------------------------------
# 6. Standardise postcodes
# -----------------------------------

df_clean["PostCode"] = (
    df_clean["PostCode"]
    .str.strip()
    .str.upper()
)


# -----------------------------------
# 7. Remove completely unusable columns
# -----------------------------------

columns_to_drop = [
    "Distance",
    "Phone",
    "RightToReply",
    "AddressLine4",
    "geocode",
    "scores"
]

df_clean = df_clean.drop(columns=columns_to_drop)


# Check the cleaned dataset
print(df_clean.shape)

df_clean.head()

(100, 24)


,AddressLine1,AddressLine2,AddressLine3,BusinessName,BusinessType,BusinessTypeID,ChangesByServerID,FHRSID,LocalAuthorityBusinessID,LocalAuthorityCode,...,PostCode,RatingDate,RatingKey,RatingValue,SchemeType,longitude,latitude,Hygiene,Structural,ConfidenceInManagement
0,NaN,NaN,NaN,(ki:ts),Retailers - other,4613,0,1916465,LKNM9H-V40BYF-WAVKXY,533,...,W1U,2026-03-04,fhrs_5_en-gb,5,FHRS,NaN,NaN,0.0,0.0,0.0
1,NaN,NaN,NaN,*Eatlife,Other catering premises,7841,0,1301468,20/00541/COMM,533,...,W1H,2025-04-17,fhrs_5_en-gb,5,FHRS,NaN,NaN,0.0,0.0,0.0
2,3 BERKELEY STREET,NaN,LONDON,1 Hotel Mayfair,Restaurant/Cafe/Canteen,1,0,1688813,GL6QH0-M4M5Q6-S02TKX,533,...,W1J 8DJ,2026-01-21,fhrs_5_en-gb,5,FHRS,-0.142200,51.507648,5.0,0.0,5.0
3,1-4 LEICESTER SQUARE,NaN,LONDON,1 Leicester Square Rooftop Bar and Restaurant,Restaurant/Cafe/Canteen,1,0,1898862,HQ202C-1TV3G5-ZN9YQ3,533,...,WC2H 7NA,2026-01-09,fhrs_5_en-gb,5,FHRS,-0.130671,51.510837,5.0,5.0,0.0
4,1 WARWICK STREET,NaN,LONDON,1 Warwick,Restaurant/Cafe/Canteen,1,0,1559410,22/00661/COMM,533,...,W1B 5LR,2025-04-29,fhrs_5_en-gb,5,FHRS,-0.137291,51.510717,0.0,0.0,0.0


## Step 19: Validate the Cleaned Dataset

After applying the cleaning methodology, the resulting dataset must be validated.

The validation checks will confirm that:

* the expected number of rows has been preserved
* duplicate establishment IDs are not present
* removed columns are no longer included
* `RatingDate` has been converted to datetime
* latitude and longitude are numeric
* missing values are still visible rather than incorrectly imputed
* the cleaned dataset remains suitable for later analysis and database loading


In [23]:
# Check dataset dimensions
print("Shape:", df_clean.shape)


# Check duplicate establishment IDs
print(
    "Duplicate FHRSID:",
    df_clean.duplicated(subset=["FHRSID"]).sum()
)


# Check data types of important transformed columns
print("\nImportant data types:")
print(
    df_clean[
        [
            "RatingDate",
            "latitude",
            "longitude",
            "Hygiene",
            "Structural",
            "ConfidenceInManagement"
        ]
    ].dtypes
)


# Check missing values
print("\nMissing values:")
print(df_clean.isna().sum().sort_values(ascending=False))


# Confirm removed columns are gone
removed_columns = [
    "Distance",
    "Phone",
    "RightToReply",
    "AddressLine4",
    "geocode",
    "scores"
]

print("\nRemoved columns still present:")
print([col for col in removed_columns if col in df_clean.columns])

Shape: (100, 24)
Duplicate FHRSID: 0

Important data types:
RatingDate                datetime64[us]
latitude                         float64
longitude                        float64
Hygiene                          float64
Structural                       float64
ConfidenceInManagement           float64
dtype: object

Missing values:
AddressLine2                  97
latitude                       6
longitude                      6
AddressLine1                   3
Structural                     3
Hygiene                        3
ConfidenceInManagement         3
AddressLine3                   3
PostCode                       1
FHRSID                         0
RatingKey                      0
BusinessName                   0
BusinessType                   0
BusinessTypeID                 0
SchemeType                     0
RatingValue                    0
RatingDate                     0
LocalAuthorityBusinessID       0
NewRatingPending               0
ChangesByServerID              0
Loc

## Step 20: Save the Cleaned Sample

The cleaned Westminster sample has passed the initial validation checks.

The processed dataset will now be saved separately from the raw API response.

The raw JSON remains unchanged in `data/raw/`, while the cleaned tabular dataset will be stored in `data/processed/`.

This cleaned sample represents the first implementation of the Stage 3 cleaning methodology and can later be translated into an automated production cleaning pipeline.


In [24]:
# Save the cleaned sample as a CSV file
df_clean.to_csv(
    "data/processed/fsa_westminster_cleaned_sample.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


In [25]:
import os

print(
    os.path.exists(
        "data/processed/fsa_westminster_cleaned_sample.csv"
    )
)

True
